# Day 9 — ILT 3: CDF-Based Incremental Loading (Bronze to Silver via MERGE)

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 2 — Watermark-Based Incremental Loading |
| **Duration** | 90 minutes |
| **Format** | Instructor-led — this enables real CDF on the real `gbmart` Bronze/Silver tables (safe, idempotent, no compute cost beyond a metadata change) |

### Learning Objectives
- Explain what Change Data Feed actually records, and why it closes the delete blind spot from ILT 2
- Enable CDF on Bronze and Silver tables the same way GlobalMart's real pipeline does
- Read a CDF changelog and understand `_change_type`, `_commit_version`, `_commit_timestamp`

---

## Why CDF, Not Just "Read The Whole Table Again"

```
Without CDF:  Silver reads all 126,000+ orders every run  →  slow, expensive, gets worse over time
With CDF:     Silver reads the ~800 rows that changed since last run  →  fast, cheap, constant cost
```

CDF is a Delta Lake table property. Once enabled, **every** `INSERT` / `UPDATE` / `DELETE` against that table is automatically recorded as a row-level event — no separate logging code needed, Delta does it as part of the transaction itself.

## What CDF Actually Records

| Column | Description |
|---|---|
| `_change_type` | `insert`, `update_preimage`, `update_postimage`, or `delete` |
| `_commit_version` | The Delta table version the change happened in |
| `_commit_timestamp` | When the change was committed |

Note the two `update_*` types — an `UPDATE` produces **two** CDF rows: the row's old values (`update_preimage`) and its new values (`update_postimage`). When you only care about "what does this row look like now," filter out `update_preimage` (you'll see this exact filter in HOL 2).

In [ ]:
# Enable CDF on all Bronze tables that are file/Autoloader-sourced. `orders` and
# `order_items` are deliberately skipped here — they're owned by the Lakeflow
# Connect pipeline (Day 2) and already use the cursor/watermark strategy from
# ILT 2 instead. This is real, idempotent — running it again does nothing harmful.
CATALOG = "gbmart"
SCHEMA  = "bronze"

BRONZE_TABLES = [
    "customers", "products", "addresses", "payments", "payment_methods", "returns"
]

for table in BRONZE_TABLES:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        spark.sql(f"ALTER TABLE {full_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        print(f"  CDF enabled : {full_name}")
    except Exception as e:
        print(f"  SKIPPED     : {full_name} — {str(e)[:80]}")

print()
print(f"  {CATALOG}.bronze.orders        <- skipped, Lakeflow Connect owns this via updated_at cursor")
print(f"  {CATALOG}.bronze.order_items   <- skipped, same reason")

In [ ]:
# Verify — DESCRIBE HISTORY will show a new version where the table property changed.
spark.sql("DESCRIBE HISTORY gbmart.bronze.customers") \
    .select("version", "timestamp", "operation") \
    .orderBy("version", ascending=False) \
    .show(5, truncate=False)

## Enabling CDF on Silver Too

Enable it now, before Silver has any incremental readers, for the same reason Bronze needed it enabled early: turning CDF on **late** means missing whatever changed in the gap. This isn't required for today's Gold layer (Day 6's dimensions and Day 7's `fact_sales` are built via full overwrite, not incremental `MERGE`) — it's about keeping the option open for later, at no cost today.

In [ ]:
SILVER_TABLES = [
    "customers", "orders", "order_items", "products",
    "address", "payments", "payment_methods", "returns"
]

for table in SILVER_TABLES:
    full_name = f"gbmart.silver.{table}"
    try:
        spark.sql(f"ALTER TABLE {full_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        print(f"  CDF enabled : {full_name}")
    except Exception as e:
        print(f"  SKIPPED     : {full_name} — {str(e)[:80]}")

## Reading a Change Feed

Once enabled, reading changes since a version is one option pair on a normal batch read — no separate API to learn.

In [ ]:
# Read every change recorded on gbmart.bronze.products since version 0 (the very
# start). In HOL 2 you'll do this starting from a real "last processed" version
# instead of 0, exactly like a production incremental job would.
cdf_df = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", 0)
        .table("gbmart.bronze.products")
)

cdf_df.groupBy("_change_type").count().show()
cdf_df.select("product_id", "discounted_price_inr", "_change_type", "_commit_version").show(10, truncate=False)

## The Bronze → Silver MERGE Pattern

This is the shape every CDF-based incremental Silver refresh follows — HOL 2 builds a real one:

1. Find the last Bronze version Silver has already processed (from a control table, same idea as ILT 2 — just tracking a Delta *version number* instead of a timestamp).
2. Read Bronze's CDF starting from `last_version + 1`.
3. Drop `update_preimage` rows — you only want each changed row's *current* state, not its "before" snapshot.
4. Apply the same cleaning/standardization transformations Day 5 used for the full load, to just this smaller changed set.
5. `MERGE` the result into Silver — update matched rows, insert new ones.
6. Record the new last-processed version.

Notice this is structurally the *same* six-step shape as ILT 2's watermark loader (read only what's new → process → write → advance the marker) — CDF just uses a Delta version number as its marker instead of a business timestamp column.